<div style="background-color:#e6f2fb; padding:20px 24px; border-radius:0;">

<h1>Homework, Task 2: Clean and pre-process the data (v4)</h1>

<p>Here we clean and pre-process train.csv for modelling. We build on the findings from Task 1 and do not repeat the analysis from there.</p>

<h2>Contents</h2>

<ol>
  <li><a href="#setup">Setup and import</a></li>
  <li><a href="#quality">Missing values and implausible values</a></li>
  <li><a href="#categorical">Categorical variables</a></li>
  <li><a href="#clientes">The optional table clientes.csv</a></li>
  <li><a href="#features">Variables and new features</a></li>
  <li><a href="#scaling">Scaling</a></li>
  <li><a href="#pipeline">Pre-processing function and comparison</a></li>
  <li><a href="#sklearn_pipeline">Formal Scikit-Learn Pipeline and ColumnTransformer</a></li>
  <li><a href="#module_and_recipe">Production module and CleaningLog recipe</a></li>
  <li><a href="#summary">Summary of decisions</a></li>
</ol>

</div>


<a id="setup"></a>
<div style="background-color:#e6f2fb; padding:20px 24px; border-radius:0;">

<h2>1. Setup and import</h2>

</div>

In [5]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, RobustScaler, MinMaxScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score

import warnings

sns.set_theme(style="whitegrid")
# no warning output, the expected one (logistic regression without log1p and scaling does not converge) is checked in section 7
warnings.filterwarnings("ignore")

# readable labels for tables and plots, the data keeps its original column names
def label(name):
    name = {"typical_gap_le_90": "typical gap at most 90 days"}.get(name, name)
    text = " ".join(str(name).replace("_", " ").split())
    return text[0].upper() + text[1:]

RANDOMSTATE = 42
TARGET = "repeat_purchase_90d"

In [6]:
trainraw = pd.read_csv("data/train.csv")
clientes = pd.read_csv("data/clientes.csv")
print("train", trainraw.shape, " clientes", clientes.shape)

FileNotFoundError: [Errno 2] No such file or directory: 'data/train.csv'

<a id="quality"></a>
<div style="background-color:#e6f2fb; padding:20px 24px; border-radius:0;">

<h2>2. Missing values and implausible values</h2>

<h3>2.1 Empty rows</h3>

</div>

In [ ]:
featurecols = [c for c in trainraw.columns if c not in ["ID", "customer_id", TARGET]]
missingperrow = trainraw[featurecols].isna().sum(axis=1)
empty = missingperrow == len(featurecols)

missingcounts = (missingperrow.value_counts().sort_index()
                 .rename_axis("Missing features per row").to_frame("Rows"))
missingcounts["Share"] = (missingcounts["Rows"] / len(trainraw)).map("{:.1%}".format)
display(missingcounts)

In [ ]:
# drop the empty rows, then split into features, target and customer groups
train = trainraw[~empty].reset_index(drop=True)
X = train.drop(columns=[TARGET])
y = train[TARGET]
groups = train["customer_id"]

pd.Series({
    "Rows left": len(train),
    "Customers": groups.nunique(),
    "Repeat rate": round(y.mean(), 3),
}, name="Value", dtype=object).rename_axis("Check").to_frame()

<div style="background-color:#e6f2fb; padding:20px 24px; border-radius:0;">

<p><b>Decision</b></p>
<ul>
  <li>We drop the <b>65 empty rows</b> from train (see Task 1, 5.1) --> there is nothing to learn from them</li>
  <li>Rows in test.csv are <b>never dropped</b>, because every ID needs a prediction</li>
  <li>There are no other duplicates (Task 1, 5.2), so nothing else is dropped</li>
</ul>

</div>

<div style="background-color:#e6f2fb; padding:20px 24px; border-radius:0;">

<h3>2.2 Repair of the implausible values</h3>

<p>In Task 1 (5.4 and 5.5) we found four kinds of implausible values: the placeholder 1e9, day counts of -4, negative amounts and values above the total spend. One repair function fixes all of them and only uses values from the same row.</p>

</div>

In [ ]:
def repair(df):
    df = df.copy()
    k = df["prior_purchase_count"]
    total = df["prior_total_spend"]

    # -4 days is a code, so it becomes missing
    for col in ["days_since_previous_purchase", "prior_mean_gap_days"]:
        df.loc[df[col] < 0, col] = np.nan

    # the mean can always be recomputed as total / count
    df["prior_mean_purchase_value"] = np.nan
    df.loc[k > 0, "prior_mean_purchase_value"] = total / k

    # a single purchase can not be bigger than the total spend
    df.loc[df["prior_max_purchase_value"] > total + 0.01, "prior_max_purchase_value"] = np.nan

    # spend_365d: placeholder to missing, remove the wrong sign, too big to missing
    df.loc[df["spend_365d"] >= 1e8, "spend_365d"] = np.nan
    df["spend_365d"] = df["spend_365d"].abs()
    df.loc[df["spend_365d"] > total + 0.01, "spend_365d"] = np.nan
    return df

In [ ]:
def countproblems(df):
    total = df["prior_total_spend"]
    return pd.Series({
        "Placeholder 1e9": ((df["prior_mean_purchase_value"] >= 1e8) | (df["spend_365d"] >= 1e8)).sum(),
        "Negative amount": ((df["prior_mean_purchase_value"] < 0) | (df["spend_365d"] < 0)).sum(),
        "Day code of minus 4": ((df["days_since_previous_purchase"] < 0) | (df["prior_mean_gap_days"] < 0)).sum(),
        "Wrong mean": ((df["prior_mean_purchase_value"] - total / df["prior_purchase_count"]).abs() > 0.02).sum(),
        "Max above total": (df["prior_max_purchase_value"] > total + 0.01).sum(),
        "Min above max": (df["prior_min_purchase_value"] > df["prior_max_purchase_value"] + 0.01).sum(),
        "Spend 365 days above total": ((df["spend_365d"] > total + 0.01) & (df["spend_365d"] < 1e8)).sum(),
    })

Xrepaired = repair(X)
pd.DataFrame({"Before": countproblems(X), "After": countproblems(Xrepaired)}).rename_axis("Problem")

In [ ]:
neg = X["spend_365d"] < 0
allinwindow = neg & (X["purchase_count_365d"] == X["prior_purchase_count"])

signcheck = pd.Series({
    "Negative spend 365 days": neg.sum(),
    "Absolute value at least spend 180 days": (X.loc[neg, "spend_365d"].abs() >= X.loc[neg, "spend_180d"]).sum(),
    "All earlier purchases inside the 365 day window": allinwindow.sum(),
    "Absolute value equal to prior total spend": np.isclose(X.loc[allinwindow, "spend_365d"].abs(),
                                                            X.loc[allinwindow, "prior_total_spend"]).sum(),
}, name="Rows").rename_axis("Check")
signcheck.to_frame()

<div style="background-color:#e6f2fb; padding:20px 24px; border-radius:0;">

<p><b>Insight</b></p>
<ul>
  <li>After the repair <b>no implausible value is left</b></li>
  <li>The mean purchase value is <b>recomputed</b> as total / count, which also fills its missing values</li>
  <li>For negative spend_365d only the <b>sign is wrong</b> --> all 15 are at least as big as spend_180d, and in 3 rows they match the total spend exactly, so we keep the absolute value</li>
  <li>All other broken values are set to <b>NaN</b> and filled in 2.4</li>
</ul>

</div>

<div style="background-color:#e6f2fb; padding:20px 24px; border-radius:0;">

<h3>2.3 Extreme values</h3>

</div>

In [ ]:
zerocost = Xrepaired["Custo"] == 0
costcheck = pd.Series({
    "Rows with cost 0": zerocost.sum(),
    "Repeat rate with cost 0": round(y[zerocost].mean(), 3),
    "Repeat rate overall": round(y.mean(), 3),
    "Rows with cost above net total": (Xrepaired["Custo"] > Xrepaired["Total Liquido"]).sum(),
    "AUC of prior total spend": round(roc_auc_score(y, Xrepaired["prior_total_spend"]), 3),
}, name="Value", dtype=object).rename_axis("Check")
display(costcheck.to_frame())

# IQR rule on the total spend, values above Q3 + 1.5 * IQR count as outliers
q1, q3 = Xrepaired["prior_total_spend"].quantile([0.25, 0.75])
upper = q3 + 1.5 * (q3 - q1)
outlier = Xrepaired["prior_total_spend"] > upper

outliercheck = pd.DataFrame({
    "Rows": [outlier.sum(), (~outlier).sum()],
    "Share": [f"{outlier.mean():.1%}", f"{(~outlier).mean():.1%}"],
    "Repeat rate": [round(y[outlier].mean(), 3), round(y[~outlier].mean(), 3)],
}, index=["Above IQR bound", "Below IQR bound"])
display(outliercheck)

<div style="background-color:#e6f2fb; padding:20px 24px; border-radius:0;">

<p><b>Decision</b></p>
<ul>
  <li>The high amounts are <b>real big customers</b> and not errors (see Task 1, 5.6)</li>
  <li>The 642 IQR outliers (9.9%) repeat <b>0.84 vs 0.53</b> --> we <b>keep</b> them and use log1p in Section 6 instead of dropping them</li>
  <li>Cost 0 (486 rows) and below cost (39 rows) can be promotions --> we keep them and add a <b>zero cost flag</b>, because these rows repeat more often (0.68 vs 0.56)</li>
</ul>

</div>

<div style="background-color:#e6f2fb; padding:20px 24px; border-radius:0;">

<h3>2.4 Missing values</h3>

<p>In Task 1 (5.3) we saw that most history values are missing on purpose (first purchases, which repeat only 33%), and the other missing values (~3%) look random.</p>

</div>

In [ ]:
# number of earlier purchases needed so that the value exists
NEEDED = {"days_since_previous_purchase": 1, "prior_mean_purchase_value": 1,
          "prior_min_purchase_value": 1, "prior_max_purchase_value": 1,
          "prior_std_purchase_value": 2, "prior_mean_gap_days": 2, "prior_std_gap_days": 3}

def fillstructural(df):
    df = df.copy()
    k = df["prior_purchase_count"]
    for col, needed in NEEDED.items():
        df.loc[k < needed, col] = 0
    df.loc[k == 0, "spend_365d"] = 0   # no earlier purchase means no spend
    df["is_first_purchase"] = (k == 0).astype(int)
    df["has_gap_history"] = (k >= 2).astype(int)
    return df


def fillmissing(train, test, plan):
    train, test = train.copy(), test.copy()
    for columns, imputer in plan:
        # numbers as float, text as object, so the imputer gets plain np.nan as missing value
        kind = float if all(pd.api.types.is_numeric_dtype(train[c]) for c in columns) else object
        blocks = [half[columns].astype(kind) for half in (train, test)]
        imputer.fit(blocks[0])
        train[columns] = imputer.transform(blocks[0])
        test[columns] = imputer.transform(blocks[1])
    return train, test

In [ ]:
Xfilled = fillstructural(Xrepaired)
missing = pd.DataFrame({"After repair": Xrepaired.isna().sum(),
                        "After structural fill": Xfilled[Xrepaired.columns].isna().sum()})
missing[missing["After repair"] > 0].sort_values("After repair", ascending=False).rename(index=label).rename_axis("Variable")

<div style="background-color:#e6f2fb; padding:20px 24px; border-radius:0;">

<p><b>Insight</b></p>
<ul>
  <li>Most gaps are <b>structural</b> --> the structural fill removes all 1413 gaps in prior_std_gap_days and leaves 209 of 812 in days_since_previous_purchase</li>
  <li>The remaining numeric gaps (45 to 231 per column) get the <b>training median</b> in Section 7</li>
  <li>Categorical columns get their own level <b>"missing"</b></li>
</ul>

</div>

<div style="background-color:#e6f2fb; padding:20px 24px; border-radius:0;">

<h3>2.5 Consistent Formats</h3>

<p>We check that every purchase date can be parsed and that no category level differs only by upper or lower case or by extra spaces.</p>

</div>

In [ ]:
print("Date parsed:", train["purchase_date"].dtype, " unparseable:", pd.to_datetime(train["purchase_date"], errors="coerce").isna().sum())
catcols = ["payment_type", "salesperson", "transport_method", "warehouse", "commercial_zone"]
pd.DataFrame({
    "Levels": [train[c].nunique() for c in catcols],
    "Levels after strip and lower": [train[c].str.strip().str.lower().nunique() for c in catcols],
}, index=[label(c) for c in catcols])

<div style="background-color:#e6f2fb; padding:20px 24px; border-radius:0;">

<p><b>Insight</b></p>
<ul>
  <li>Every purchase date can be parsed and no level differs only by case or spaces --> <b>no format fixes needed</b></li>
  <li>The date is stored as text and is converted where it is used</li>
</ul>

</div>

<a id="categorical"></a>
<div style="background-color:#e6f2fb; padding:20px 24px; border-radius:0;">

<h2>3. Categorical variables</h2>

<table>
  <tr><th>Variable</th><th>Decision</th><th>Why</th></tr>
  <tr><td>Payment type, salesperson</td><td>one-hot</td><td>the repeat rate differs clearly between levels (Task 1, 8.3)</td></tr>
  <tr><td>Transport method</td><td>one-hot, rare levels grouped</td><td>80 levels and 51 of them with less than 30 rows (Task 1, 4.2)</td></tr>
  <tr><td>Commercial zone</td><td>one-hot</td><td>weak, Task 3 can still remove it</td></tr>
  <tr><td>Warehouse</td><td>drop</td><td>99.7% one level (Task 1, 4.2)</td></tr>
  <tr><td>Document series</td><td>drop</td><td>no link with the target and each series only covers some years (Task 1, 8.3 and 10)</td></tr>
  <tr><td>Purchase date</td><td>keep only the month as a category</td><td>the repeat rate changes a lot by month (see below), the year is dropped because later purchases fall into years the model has never seen</td></tr>
</table>

<ul>
  <li>We use a OneHotEncoder with a <b>minimum frequency of 30</b> --> rare levels go into one "infrequent" column</li>
  <li>Levels that do not occur in the training rows also go there instead of giving an error</li>
  <li>No variable has a natural order, so we do not need ordinal encoding</li>
</ul>

</div>

In [ ]:
month = pd.to_datetime(X["purchase_date"]).dt.month
ratebymonth = y.groupby(month).mean()

ax = ratebymonth.plot(kind="bar", figsize=(10, 3))
ax.axhline(y.mean(), color="grey", linestyle="--")
ax.set(title="Repeat rate by month", xlabel="Month", ylabel="Repeat rate")
plt.show()
# every row counts once, so customers with many purchases dominate, the second row first averages per customer
percustomer = y.groupby([groups, month]).mean().groupby(level=1).mean()
pd.DataFrame({"Repeat rate per row": ratebymonth, "Repeat rate per customer": percustomer}).rename_axis("Month").round(2).T

In [ ]:
# same monthly pattern in the first and the second half of the years in train?
year = pd.to_datetime(X["purchase_date"]).dt.year
early = year <= 2019

monthcheck = pd.DataFrame({
    "2015 to 2019": y[early].groupby(month[early]).mean(),
    "2020 to 2025": y[~early].groupby(month[~early]).mean(),
}).round(3).rename_axis("Month")
display(monthcheck)
print("Correlation of the two periods:", round(monthcheck["2015 to 2019"].corr(monthcheck["2020 to 2025"]), 2))

<div style="background-color:#e6f2fb; padding:20px 24px; border-radius:0;">

<p><b>Insight</b></p>
<ul>
  <li>The repeat rate changes a lot by month, from <b>0.44 in March</b> to <b>0.72 in May</b></li>
  <li>The pattern is <b>stable</b> --> 2015 to 2019 and 2020 to 2025 correlate at 0.81</li>
  <li>Per customer the rates are lower (March 0.31, May 0.61), because frequent buyers dominate the rows, but the pattern stays</li>
  <li>We keep the <b>month</b> and drop the <b>year</b>, because later purchases fall into years the model has never seen. Section 7.3 tests the month on later years</li>
</ul>

</div>

In [ ]:
def addmonth(df):
    df = df.copy()
    # month as text, so the encoder treats it as a category and not as a number
    df["month"] = pd.to_datetime(df["purchase_date"]).dt.strftime("%b")
    return df

CATEGORICAL = ["payment_type", "salesperson", "commercial_zone", "transport_method", "month"]
Xmonth = addmonth(X)

# levels under 30 rows and unseen levels both go into one "infrequent" column
encoder = OneHotEncoder(min_frequency=30, handle_unknown="infrequent_if_exist", sparse_output=False)
encoder.fit(Xmonth[CATEGORICAL].fillna("missing"))

# how many columns each variable turns into after grouping
onehotnames = encoder.get_feature_names_out(CATEGORICAL)
encodingsummary = pd.DataFrame({
    "Levels": [Xmonth[col].nunique() for col in CATEGORICAL],
    "One hot columns": [sum(name.startswith(col + "_") for name in onehotnames) for col in CATEGORICAL],
}, index=[label(col) for col in CATEGORICAL]).rename_axis("Variable")
display(encodingsummary)
print("One hot columns in total:", len(onehotnames))

<a id="clientes"></a>
<div style="background-color:#e6f2fb; padding:20px 24px; border-radius:0;">

<h2>4. The optional table clientes.csv</h2>

<ul>
  <li>In Task 1 (6) we saw that most columns of clientes are empty, identifiers or have too many levels</li>
  <li>Only <b>country_or_region</b> and <b>whether the island is known</b> looked interesting --> both are known at the time of a purchase, so there is no leakage</li>
  <li>We join them with a <b>left join</b> by customer_id, so no purchase is lost, and test in Section 7 if they help</li>
</ul>

</div>

In [ ]:
def joinclientes(df):
    extra = clientes[["customer_id", "country_or_region"]].copy()
    extra["island_known"] = clientes["island"].notna().astype(int)
    out = df.merge(extra, on="customer_id", how="left")
    out.index = df.index
    return out

<a id="features"></a>
<div style="background-color:#e6f2fb; padding:20px 24px; border-radius:0;">

<h2>5. Variables and new features</h2>

<h3>5.1 Supplied variables</h3>

<table>
  <tr><th>Variable</th><th>Decision</th><th>Why</th></tr>
  <tr><td>ID</td><td>not used, only for the submission</td><td>no signal (Task 1, 5.7)</td></tr>
  <tr><td>Customer id</td><td>not used, only for the join and the folds</td><td>the model would learn the customers by heart</td></tr>
  <tr><td>Random noise</td><td>drop</td><td>only noise (Task 1, 5.7)</td></tr>
  <tr><td>Total a Pagar, Total Bruto</td><td>drop</td><td>can be calculated from other columns (Task 1, 5.5)</td></tr>
  <tr><td>all other numeric columns</td><td>keep</td><td>Task 3 selects the features</td></tr>
</table>

<h3>5.2 New features</h3>

<ul>
  <li>The new features only use values from the <b>same row</b> --> they contain nothing from later purchases</li>
  <li>We do not build our own customer totals, because the history columns already are totals up to the purchase</li>
</ul>

<table>
  <tr><th>Feature</th><th>How</th><th>Idea</th></tr>
  <tr><td>Purchases per year</td><td>prior purchases / tenure in years</td><td>how often the customer buys</td></tr>
  <tr><td>Typical gap at most 90 days</td><td>tenure / prior purchases ≤ 90</td><td>rough average time between purchases</td></tr>
  <tr><td>Discount rate</td><td>(line + global discount) / gross total</td><td>share of the price given as discount</td></tr>
  <tr><td>Zero cost</td><td>cost = 0</td><td>these rows repeat more often (2.3)</td></tr>
  <tr><td>Has shipping</td><td>shipping cost > 0</td><td>delivered or picked up</td></tr>
</table>

</div>

In [ ]:
def addfeatures(df):
    df = df.copy()
    k = df["prior_purchase_count"]
    tenure = df["customer_tenure_days"]

    df["purchases_per_year"] = 0.0
    df.loc[tenure > 0, "purchases_per_year"] = k / (tenure / 365)

    meangap = tenure / k.replace(0, np.nan)   # average days per earlier purchase, not the exact gap
    df["typical_gap_le_90"] = (meangap <= 90).astype(int)

    # gross total is never 0 in train, the guard keeps new data with a 0 from dividing by 0
    df["discount_rate"] = ((df["Total Desc_ Linha"] + df["Total Desc_ Global"]) / df["Total Bruto"].replace(0, np.nan)).fillna(0)
    df["zero_cost"] = (df["Custo"] == 0).astype(int)
    df["has_shipping"] = (df["Total Portes"] > 0).astype(int)
    return df

Xfeatures = addfeatures(Xfilled)

In [ ]:
newfeatures = ["purchases_per_year", "typical_gap_le_90", "discount_rate", "zero_cost", "has_shipping"]
# supplied columns for comparison
reference = ["purchase_count_365d", "prior_purchase_count", "Total Liquido", "Total Desc_ Linha", "Total Desc_ Global", "Custo", "N_ Detalhes"]
pd.Series({col: roc_auc_score(y, Xfeatures[col]) for col in newfeatures + reference}).rename(label).rename_axis("Feature").to_frame("AUC").round(3)

<div style="background-color:#e6f2fb; padding:20px 24px; border-radius:0;">

<p><b>Insight</b></p>
<ul>
  <li><b>Purchases per year</b> (AUC 0.744) is almost as strong as purchase_count_365d (0.753) and stronger than prior_purchase_count (0.701)</li>
  <li><b>Typical gap at most 90 days</b> (0.692) turns the 90 day rule of the target into a feature</li>
  <li><b>Discount rate</b> (0.624) is stronger than every invoice column (best: Total Desc_ Linha with 0.611)</li>
  <li>Zero cost and has shipping are <b>weak</b> on their own --> Task 3 decides if they stay</li>
</ul>

</div>

<a id="scaling"></a>
<div style="background-color:#e6f2fb; padding:20px 24px; border-radius:0;">

<h2>6. Scaling</h2>

<h3>6.1 log1p for the skewed columns</h3>

<p>Most amounts and counts are strongly right skewed (Task 1, 4.1). We use log1p because it also works for 0.</p>

</div>

In [ ]:
NUMERIC = ["Total Liquido", "Total Imp_", "Total Portes", "Total Desc_ Global", "Total Desc_ Linha", "Custo",
           "N_ Detalhes", "invoice_count_current_purchase",
           "days_since_previous_purchase", "customer_tenure_days", "prior_purchase_count", "prior_total_spend",
           "prior_mean_purchase_value", "prior_std_purchase_value", "prior_min_purchase_value",
           "prior_max_purchase_value", "prior_mean_gap_days", "prior_std_gap_days",
           "purchase_count_30d", "spend_30d", "purchase_count_90d", "spend_90d",
           "purchase_count_180d", "spend_180d", "purchase_count_365d", "spend_365d"]

# log1p is only defined for values above -1
belowminusone = (Xfeatures[NUMERIC] < -1).sum()
print("Columns with values below -1:", (belowminusone > 0).sum())

skew = pd.DataFrame({"Skew before": Xfeatures[NUMERIC].skew(),
                     "Skew after log1p": np.log1p(Xfeatures[NUMERIC]).skew()})
print("Median absolute skew before:", round(skew["Skew before"].abs().median(), 2),
      " after:", round(skew["Skew after log1p"].abs().median(), 2))
skew.index = skew.index.map(label)
skew.sort_values("Skew before", ascending=False).head(8).rename_axis("Variable").round(2)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 6))
for j, col in enumerate(["prior_total_spend", "days_since_previous_purchase"]):
    sns.histplot(Xfeatures[col], bins=50, ax=axes[0, j])
    axes[0, j].set(title=label(col), xlabel=label(col), ylabel="Count")
    sns.histplot(np.log1p(Xfeatures[col]), bins=50, ax=axes[1, j])
    axes[1, j].set(title="log1p of " + label(col).lower(), xlabel="log1p of " + label(col).lower(), ylabel="Count")
plt.tight_layout()
plt.show()

<div style="background-color:#e6f2fb; padding:20px 24px; border-radius:0;">

<p><b>Insight</b></p>
<ul>
  <li>log1p brings the median absolute skew down from <b>3.7 to about 1</b> --> the columns become much more symmetric</li>
  <li>Total Desc_ Global stays skewed because it is almost always 0, and the invoice count because almost every purchase has only one invoice</li>
</ul>

</div>

<div style="background-color:#e6f2fb; padding:20px 24px; border-radius:0;">

<h3>6.2 Scaler</h3>

<ul>
  <li>Scaling matters for <b>logistic regression</b>: the regularisation treats all coefficients the same and the solver converges better</li>
  <li>A <b>decision tree</b> only splits on thresholds and does not need it</li>
  <li>Section 7 compares no scaler, StandardScaler, RobustScaler and MinMaxScaler, each with and without log1p</li>
  <li>Flags and one-hot columns are already 0 or 1 and are not scaled</li>
</ul>

</div>

<a id="pipeline"></a>
<div style="background-color:#e6f2fb; padding:20px 24px; border-radius:0;">

<h2>7. Pre-processing function and comparison</h2>

<h3>7.1 One function for all steps</h3>

<p>All steps go into one function. It fits everything on train and only transforms test --> in Task 4 we call it with train.csv and test.csv.</p>

</div>

In [ ]:
SCALERS = {"none": None, "standard": StandardScaler, "robust": RobustScaler, "minmax": MinMaxScaler}

def foldmatrices(train, test, usemonth=True, usefeatures=True, useclientes=False, uselog=True, scaler="standard",
                 userepair=True, usestructural=True):
    numeric = list(NUMERIC)
    categorical = list(CATEGORICAL)
    if not usemonth:
        categorical.remove("month")
    flags = []

    # row rules, safe to apply to train and test alike
    if userepair:
        train, test = repair(train), repair(test)
    if usestructural:
        train, test = fillstructural(train), fillstructural(test)
        flags += ["is_first_purchase", "has_gap_history"]
    train, test = addmonth(train), addmonth(test)
    if usefeatures:
        train, test = addfeatures(train), addfeatures(test)
        numeric += ["purchases_per_year", "discount_rate"]
        flags += ["typical_gap_le_90", "zero_cost", "has_shipping"]
    if useclientes:
        train, test = joinclientes(train), joinclientes(test)
        categorical += ["country_or_region"]
        flags += ["island_known"]

    # imputers are fitted on train only
    train, test = fillmissing(train, test, [
        (numeric, SimpleImputer(strategy="median")),
        (categorical, SimpleImputer(strategy="constant", fill_value="missing")),
    ])

    numtrain = train[numeric].to_numpy(dtype=float)
    numtest = test[numeric].to_numpy(dtype=float)
    if uselog:
        # discount_rate is a share between 0 and 1, so no log
        logcols = [i for i, c in enumerate(numeric) if c != "discount_rate"]
        # clip only matters in the ablation without repair, where -4 codes and negative amounts remain
        numtrain[:, logcols] = np.log1p(np.clip(numtrain[:, logcols], 0, None))
        numtest[:, logcols] = np.log1p(np.clip(numtest[:, logcols], 0, None))
    if SCALERS[scaler] is not None:
        fitted = SCALERS[scaler]().fit(numtrain)
        numtrain, numtest = fitted.transform(numtrain), fitted.transform(numtest)

    # flags and one-hot columns are already 0 or 1, so they are not scaled
    encoder = OneHotEncoder(min_frequency=30, handle_unknown="infrequent_if_exist", sparse_output=False)
    cattrain = encoder.fit_transform(train[categorical])
    cattest = encoder.transform(test[categorical])

    Atrain = np.hstack([numtrain, train[flags].to_numpy(dtype=float), cattrain])
    Atest = np.hstack([numtest, test[flags].to_numpy(dtype=float), cattest])
    return Atrain, Atest

<div style="background-color:#e6f2fb; padding:20px 24px; border-radius:0;">

<p>As a comparison we also build a naive version without repair, without the flags and without new features, with the 1e9 codes still in the data.</p>

</div>

In [ ]:
NAIVENUMERIC = NUMERIC + ["Total a Pagar", "Total Bruto"]
NAIVECATEGORICAL = ["payment_type", "salesperson", "commercial_zone", "transport_method", "warehouse", "document_series"]

def naivematrices(train, test):
    train, test = fillmissing(train, test, [
        (NAIVENUMERIC, SimpleImputer(strategy="median")),
        (NAIVECATEGORICAL, SimpleImputer(strategy="constant", fill_value="missing")),
    ])
    sc = StandardScaler().fit(train[NAIVENUMERIC])
    enc = OneHotEncoder(handle_unknown="ignore", sparse_output=False).fit(train[NAIVECATEGORICAL])
    Atrain = np.hstack([sc.transform(train[NAIVENUMERIC]), enc.transform(train[NAIVECATEGORICAL])])
    Atest = np.hstack([sc.transform(test[NAIVENUMERIC]), enc.transform(test[NAIVECATEGORICAL])])
    return Atrain, Atest

<div style="background-color:#e6f2fb; padding:20px 24px; border-radius:0;">

<h3>7.2 Comparison on held-out customers</h3>

<ul>
  <li>We split the <b>customers</b> into 5 fixed folds --> the same customer is never in train and validation</li>
  <li>A random split would look too good, because the rows of one customer share their history (Task 1, 9.3)</li>
  <li>We score with <b>ROC AUC</b> using a logistic regression and a decision tree with a maximum depth of 5</li>
</ul>

</div>

In [ ]:
# own grouped folds: shuffle the customers once and split them into 5 groups,
# so the folds are the same in every scikit-learn version
customers = np.random.default_rng(RANDOMSTATE).permutation(groups.unique())
foldofcustomer = {c: i % 5 for i, c in enumerate(customers)}
foldid = groups.map(foldofcustomer).to_numpy()
splits = [(np.where(foldid != k)[0], np.where(foldid == k)[0]) for k in range(5)]

models = {"Logistic Regression": LogisticRegression(max_iter=2000),
          "Decision Tree": DecisionTreeClassifier(max_depth=5, random_state=RANDOMSTATE)}

def heldoutauc(prepare, model):
    # prepare and fit on the training customers, score on the unseen customers
    scores = []
    for trainidx, testidx in splits:
        Atrain, Atest = prepare(X.iloc[trainidx], X.iloc[testidx])
        model.fit(Atrain, y.iloc[trainidx])
        scores.append(roc_auc_score(y.iloc[testidx], model.predict_proba(Atest)[:, 1]))
    return np.array(scores)

In [ ]:
variants = {
    "A naive": naivematrices,
    "B cleaned": lambda tr, te: foldmatrices(tr, te, usemonth=False, usefeatures=False),
    "C B + month": lambda tr, te: foldmatrices(tr, te, usefeatures=False),
    "D C + new features": lambda tr, te: foldmatrices(tr, te),
    "E D + clientes": lambda tr, te: foldmatrices(tr, te, useclientes=True),
}

scores = {}
for name, prepare in variants.items():
    for modelname, model in models.items():
        scores[(name, modelname)] = heldoutauc(prepare, model)

pd.Series({key: s.mean() for key, s in scores.items()}).unstack().rename_axis(index="Variant", columns="Model").round(4)

In [ ]:
# difference to the step before, on the same folds
names = list(variants)
for modelname in models:
    print(modelname)
    for before, after in zip(names, names[1:]):
        diff = scores[(after, modelname)] - scores[(before, modelname)]
        print(f"  {after[:1]} vs {before[:1]}: {diff.mean():+.4f}, better in {(diff > 0).sum()} of 5 folds")

<div style="background-color:#e6f2fb; padding:20px 24px; border-radius:0;">

<p>The jump from A to B bundles several steps. To see which one matters we remove one step at a time from B and compare on the same folds.</p>

</div>

In [ ]:
ablation = {
    "B cleaned": {},
    "B without repair": {"userepair": False},
    "B without structural fill and flags": {"usestructural": False},
    "B without log1p": {"uselog": False},
}
rows = {}
for name, switches in ablation.items():
    for modelname, model in models.items():
        s = heldoutauc(lambda tr, te: foldmatrices(tr, te, usemonth=False, usefeatures=False, **switches), model)
        base = scores[("B cleaned", modelname)]
        rows[(name, modelname)] = f"{s.mean():.4f} ({s.mean() - base.mean():+.4f}, better in {(s > base).sum()} of 5)"
pd.Series(rows).unstack().rename_axis(index="Variant", columns="Model")

In [ ]:
scaling = {}
for scaler in ["none", "standard", "robust", "minmax"]:
    for withlog in [False, True]:
        s = heldoutauc(lambda tr, te: foldmatrices(tr, te, uselog=withlog, scaler=scaler),
                         models["Logistic Regression"])
        scaling[(scaler.capitalize(), "With log1p" if withlog else "Without log1p")] = s.mean()

pd.Series(scaling).unstack().rename_axis(index="Scaler", columns="Transform").round(4)

In [ ]:
# without log1p and scaler lbfgs stops at max_iter, so we check how often and repeat with a solver that converges
Atr, Ava = foldmatrices(X.iloc[splits[0][0]], X.iloc[splits[0][1]], uselog=False, scaler="none")
lbfgs = LogisticRegression(max_iter=2000).fit(Atr, y.iloc[splits[0][0]])
print("Iterations used in fold 1:", lbfgs.n_iter_[0], "of 2000")

s = heldoutauc(lambda tr, te: foldmatrices(tr, te, uselog=False, scaler="none"),
                 LogisticRegression(solver="newton-cholesky", max_iter=2000))
print("No log1p, no scaler, newton-cholesky solver:", round(s.mean(), 4))
print("Spread of the 5 fold scores of D:", round(scores[("D C + new features", "Logistic Regression")].std(), 4))

<div style="background-color:#e6f2fb; padding:20px 24px; border-radius:0;">

<h3>7.3 Check over time</h3>

<ul>
  <li>The grouped folds mix all years, but later data comes from <b>later years</b></li>
  <li>So we also train on all years before a year and validate on that year (2018 to 2024) with the logistic regression</li>
</ul>

</div>

In [ ]:
year = pd.to_datetime(X["purchase_date"]).dt.year
rows = {}
for valyear in range(2018, 2025):
    tr, va = year < valyear, year == valyear
    for name in ["B cleaned", "C B + month", "D C + new features", "E D + clientes"]:
        Atr, Ava = variants[name](X[tr], X[va])
        model = LogisticRegression(max_iter=2000).fit(Atr, y[tr])
        rows[(valyear, name)] = roc_auc_score(y[va], model.predict_proba(Ava)[:, 1])

timescores = pd.Series(rows).unstack()
gains = pd.DataFrame({"Month (C vs B)": timescores["C B + month"] - timescores["B cleaned"],
                      "New features (D vs C)": timescores["D C + new features"] - timescores["C B + month"],
                      "Clientes (E vs D)": timescores["E D + clientes"] - timescores["D C + new features"]})
gains.loc["Mean"] = gains.mean()
gains.rename_axis("Validation year").round(4)

<div style="background-color:#e6f2fb; padding:20px 24px; border-radius:0;">

<p><b>Insight on clientes.csv</b></p>
<ul>
  <li>The two clientes columns add <b>almost nothing</b>: +0.0003 for the logistic regression, -0.0019 for the tree and +0.0007 over time</li>
  <li>This is far below the spread between the folds --> we <b>do not use clientes.csv</b>, but keep the join function in the code</li>
</ul>

</div>

<a id="sklearn_pipeline"></a>
<div style="background-color:#e6f2fb; padding:20px 24px; border-radius:0;">

<h2>8. Formal Scikit-Learn Pipeline & ColumnTransformer</h2>

<p>While <code>foldmatrices()</code> in Section 7 cleanly proves our transformations across customer folds, standard Scikit-Learn feature selection tools (Task 3: <code>RFE</code>, <code>SequentialFeatureSelector</code>) and hyperparameter search engines (Task 4: <code>GridSearchCV</code>) require standard Scikit-Learn estimators implementing <code>fit()</code> and <code>transform()</code>.</p>

<ul>
  <li>We encapsulate row repairs, structural fills, and feature engineering inside a custom <b><code>DomainPreprocessor</code></b> transformer (stateless).</li>
  <li>We assemble a <b><code>ColumnTransformer</code></b> with separate branches for log-skewed numerics, non-log numerics (<code>discount_rate</code>), binary flags, and categoricals (OneHotEncoder).</li>
  <li>We verify that the Scikit-Learn Pipeline produces <b>100% identical outputs</b> to <code>foldmatrices()</code> down to float precision, while supporting <code>get_feature_names_out()</code>.</li>
</ul>

</div>

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer

# 1. Custom stateless transformer for row-wise repairs and engineered features
class DomainPreprocessor(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        X = repair(X)
        X = fillstructural(X)
        X = addmonth(X)
        X = addfeatures(X)
        return X

# 2. Column groups
log_numeric = NUMERIC + ["purchases_per_year"]
plain_numeric = ["discount_rate"]
flags = ["is_first_purchase", "has_gap_history", "typical_gap_le_90", "zero_cost", "has_shipping"]
cats = list(CATEGORICAL)

# 3. Unified ColumnTransformer with named outputs
col_transformer = ColumnTransformer(
    transformers=[
        ("log_num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("log1p", FunctionTransformer(lambda x: np.log1p(np.clip(x, 0, None)), feature_names_out="one-to-one")),
            ("scaler", StandardScaler()),
        ]), log_numeric),
        ("plain_num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]), plain_numeric),
        ("flags", "passthrough", flags),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
            ("ohe", OneHotEncoder(min_frequency=30, handle_unknown="infrequent_if_exist", sparse_output=False)),
        ]), cats),
    ],
    remainder="drop"
)

full_preprocessor = Pipeline([
    ("domain", DomainPreprocessor()),
    ("columns", col_transformer),
])


In [ ]:
# Verify mathematical equivalence between foldmatrices and the formal Pipeline
tr_idx, te_idx = splits[0]
A_tr_fold, A_te_fold = foldmatrices(X.iloc[tr_idx], X.iloc[te_idx])

A_tr_pipe = full_preprocessor.fit_transform(X.iloc[tr_idx])
A_te_pipe = full_preprocessor.transform(X.iloc[te_idx])

max_diff_train = np.max(np.abs(A_tr_fold - A_tr_pipe))
max_diff_val = np.max(np.abs(A_te_fold - A_te_pipe))

feature_names = [f.split("__")[-1] for f in col_transformer.get_feature_names_out()]

pd.Series({
    "Pipeline output shape (Train)": str(A_tr_pipe.shape),
    "Pipeline output shape (Val)": str(A_te_pipe.shape),
    "Max absolute difference vs foldmatrices (Train)": f"{max_diff_train:.2e}",
    "Max absolute difference vs foldmatrices (Val)": f"{max_diff_val:.2e}",
    "Total named features extracted": len(feature_names),
    "Identical within float precision": np.allclose(A_tr_fold, A_tr_pipe, atol=1e-10),
}, name="Pipeline Verification").to_frame()

<a id="module_and_recipe"></a>
<div style="background-color:#e6f2fb; padding:20px 24px; border-radius:0;">

<h2>9. Production Module & CleaningLog Recipe</h2>

<p>Following the course standard established in Week 3 (<code>course_helpers.py</code>), we record all cleaning decisions into an immutable <b><code>CleaningLog</code></b> and export both the audit trail (<code>cleaning_recipe.json</code>) and a standalone Python module (<code>preprocess_pipeline.py</code>).</p>

<ul>
  <li><b>Single source of truth</b>: Task 3 (Feature Selection) and Task 4 (Modelling) can ingest clean data with one line: <code>from preprocess_pipeline import get_clean_data</code>.</li>
  <li><b>Auditability</b>: Every data modification, row drop, sentinel replacement, and transformation is permanently documented with stated reasons and row impacts in JSON format.</li>
</ul>

</div>

In [ ]:
import sys
from pathlib import Path
if "." not in sys.path:
    sys.path.insert(0, ".")

from course_helpers import CleaningLog, CleaningStep
from preprocess_pipeline import build_cleaning_log, save_cleaning_recipe, get_clean_data

# Build the course CleaningLog and save to cleaning_recipe.json
log = build_cleaning_log()
recipe_path = save_cleaning_recipe("cleaning_recipe.json")

print(f"Recorded {len(log)} cleaning steps in {recipe_path}")
display(pd.DataFrame(log.records()).rename(columns={
    "column": "Column / Scope",
    "action": "Action Taken",
    "reason": "Domain / Methodological Reason",
    "rows_affected": "Rows Affected"
}))


In [ ]:
# Demonstration: 1-line clean data loading for Task 3 and Task 4
testraw = pd.read_csv("data/test.csv")

X_train_clean, X_test_clean, y_train_clean, feature_cols = get_clean_data(trainraw, testraw, return_df=True)

pd.Series({
    "X_train clean shape": str(X_train_clean.shape),
    "X_test clean shape": str(X_test_clean.shape),
    "y_train shape": str(y_train_clean.shape),
    "Total feature columns": len(feature_cols),
    "Any NaNs in X_train": X_train_clean.isna().any().any(),
    "Any NaNs in X_test": X_test_clean.isna().any().any(),
}, name="Production Data Verification").to_frame()


<a id="summary"></a>
<div style="background-color:#e6f2fb; padding:20px 24px; border-radius:0;">

<h2>10. Summary of decisions</h2>

<table>
  <tr><th>Problem / Step</th><th>Decision</th><th>Course Rationale</th></tr>
  <tr><td>65 empty rows</td><td>Dropped from train only; test rows never dropped</td><td>Uninformative observations; zero test leakage</td></tr>
  <tr><td>Entity repeated measures</td><td>Grouped 5-fold cross-validation by customer_id</td><td>Prevents customer identity leakage across train and val</td></tr>
  <tr><td>Sentinel -4 & placeholders</td><td>Blanked to NaN; recomputed mean as total/count; abs() for negative spend</td><td>Physical domain rules vs statistical noise (Week 3)</td></tr>
  <tr><td>High-spend IQR outliers (642 rows)</td><td>Retained 100% of rows; variance stabilized with log1p</td><td>Outliers repeat at 84% vs 53%; dropping deletes top customers (Week 3)</td></tr>
  <tr><td>Structural missingness</td><td>Filled with 0 + is_first_purchase & has_gap_history flags</td><td>Structural characteristics of first-time buyers, not random missingness</td></tr>
  <tr><td>Random missingness</td><td>Median for numeric; 'missing' category level for text</td><td>Domain benchmark; robust to non-normality (Week 3)</td></tr>
  <tr><td>Categorical encoding</td><td>OneHotEncoder(min_frequency=30, handle_unknown='infrequent_if_exist')</td><td>Collapses rare levels to prevent dimensional explosion (Week 4)</td></tr>
  <tr><td>Warehouse & series</td><td>Dropped warehouse (99.7% constant) and document_series</td><td>Near-zero variance and time-bound noise removal (Week 4)</td></tr>
  <tr><td>Purchase date</td><td>Extracted month as categorical; dropped year</td><td>Captures stable seasonality (r=0.81); prevents out-of-time year drift (Week 5)</td></tr>
  <tr><td>Auxiliary table clientes.csv</td><td>Discarded (gain far below fold variance: +0.0003 LogReg, -0.0019 Tree)</td><td>Empirically tested; Occam's razor (Week 2 & 5)</td></tr>
  <tr><td>Feature engineering</td><td>5 domain interaction ratios & flags (purchases_per_year, typical_gap_le_90, etc.)</td><td>Domain ratios expose non-linear relationships to linear models (Week 4)</td></tr>
  <tr><td>Feature scaling</td><td>log1p + StandardScaler on all numeric columns</td><td>Reduces skew from 3.7 to 0.99; ensures gradient convergence (Week 3)</td></tr>
  <tr><td>Architecture & reproducibility</td><td>Scikit-Learn Pipeline + ColumnTransformer + CleaningLog recipe</td><td>Full modularity for Task 3 (Feature Selection) & Task 4 (GridSearchCV)</td></tr>
</table>

</div>
